<a href="https://colab.research.google.com/github/Mon3em20/deep-learning-image-captioning/blob/main/image_captioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Person 1 code

In [ ]:
# ==========================================
# 0. Environment Setup
# ==========================================
!pip install -q kaggle pycocotools tqdm

# Move kaggle.json into place (upload it to /content/ first!)
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp /content/kaggle.json /root/.kaggle/ 2>/dev/null && echo "✅ kaggle.json copied"
!chmod 600 /root/.kaggle/kaggle.json 2>/dev/null

print("✅ Environment ready.")

✅ Environment ready.


In [ ]:
# ==========================================
# 1. Download & Extract the Dataset (all remote – no local download)
# ==========================================
import os, zipfile, glob

os.makedirs('/content/coco', exist_ok=True)
%cd /content/coco

# Download the main zip from Kaggle
!kaggle datasets download -d hariwh0/ms-coco-dataset
print("✅ Main archive downloaded.")

# Unzip
with zipfile.ZipFile('ms-coco-dataset.zip', 'r') as z:
    z.extractall('.')
!rm -f ms-coco-dataset.zip
print("✅ Main archive extracted.")

# The archive contains nested zip files: train2014.zip, val2014.zip, captions.zip
for f in ['train2014.zip', 'val2014.zip', 'captions.zip']:
    if os.path.exists(f):
        print(f"Unzipping {f} ...")
        with zipfile.ZipFile(f, 'r') as z:
            z.extractall('.')
        os.remove(f)

print("\n✅ All data extracted. Contents:")
!ls -F /content/coco/

/content/coco
Dataset URL: https://www.kaggle.com/datasets/hariwh0/ms-coco-dataset
License(s): unknown
100% 12.7G/12.7G [01:58<00:00, 116MB/s]

✅ Main archive downloaded.
✅ Main archive extracted.

✅ All data extracted. Contents:
captions/  train2014/


In [ ]:
# ==========================================
# 2. Person 1 – Load & Clean Captions
# ==========================================
import json, string, glob

# Locate the training captions JSON
ann_dir = '/content/coco/captions/annotations'
train_ann = glob.glob(os.path.join(ann_dir, '*captions_train20*.json'))
if not train_ann:
    raise FileNotFoundError("No captions_train20XX.json found.")
train_ann = train_ann[0]
print(f"Using captions file: {train_ann}")

with open(train_ann, 'r') as f:
    data = json.load(f)

annotations = data['annotations']
print(f"Loaded {len(annotations)} captions.")

def clean_caption(caption):
    """Matching the PDF: lower, remove punctuation/numbers, add startseq/endseq."""
    caption = caption.lower()
    caption = caption.translate(str.maketrans('', '', string.punctuation))
    caption = ''.join(ch for ch in caption if not ch.isdigit())
    caption = ' '.join(caption.split())
    return 'startseq ' + caption + ' endseq'

all_cleaned_captions = []
for ann in annotations:
    cap = ann['caption'].strip()
    if cap:
        all_cleaned_captions.append(clean_caption(cap))

print(f"Total cleaned captions: {len(all_cleaned_captions)}")
print("Sample:", all_cleaned_captions[0])

Using captions file: /content/coco/captions/annotations/captions_train2014.json
Loaded 414113 captions.
Total cleaned captions: 414113
Sample: startseq a very clean and well decorated empty bathroom endseq


In [ ]:
# ==========================================
# 3. Person 1 – Tokenizer, Vocabulary, Save
# ==========================================
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer

# Build tokenizer
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(all_cleaned_captions)

vocab_size = len(tokenizer.word_index) + 1  # +1 for padding (0)
print(f"Vocabulary size (with padding): {vocab_size}")

# Max caption length
all_sequences = tokenizer.texts_to_sequences(all_cleaned_captions)
valid_sequences = [s for s in all_sequences if len(s) >= 2]  # need start+end
max_sequence_length = max(len(s) for s in valid_sequences)

# Save tokenizer and max length (required by Person 3)
with open('/content/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
with open('/content/max_sequence_length.txt', 'w') as f:
    f.write(str(max_sequence_length))

print(f"Max caption sequence length: {max_sequence_length}")
print("✅ Saved: tokenizer.pkl, max_sequence_length.txt")

Vocabulary size (with padding): 24408
Max caption sequence length: 51
✅ Saved: tokenizer.pkl, max_sequence_length.txt


In [ ]:
# ==========================================
# 4. Person 1 – Padded Sequences & Train/Val Split (Optional)
# ==========================================
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np

X, y = [], []
for seq in valid_sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

X_padded = pad_sequences(X, maxlen=max_sequence_length, padding='pre')
y = np.array(y, dtype=np.int32)

X_train, X_val, y_train, y_val = train_test_split(
    X_padded, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")

np.save('/content/X_train.npy', X_train)
np.save('/content/y_train.npy', y_train)
np.save('/content/X_val.npy', X_val)
np.save('/content/y_val.npy', y_val)
print("✅ Saved X_train, y_train, X_val, y_val.")

Train: (3794426, 51), Val: (948607, 51)
✅ Saved X_train, y_train, X_val, y_val.


# Person 2 Part of code


In [ ]:
# ==========================================
# 5. Person 2 – Imports, Device, Auto‑detect Image Folders, Filter Existing
# ==========================================
import torch
import torchvision
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import json, random, glob, os
from PIL import Image
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Device:", device)

# --- Helper: find folder containing .jpg files ---
def find_image_dir(base_dir, keyword='train'):
    """Search recursively for a folder that contains .jpg files and has 'keyword' in its path."""
    candidates = []
    for root, dirs, files in os.walk(base_dir):
        if any(f.endswith('.jpg') for f in files):
            candidates.append(root)
    # Prefer one that contains the keyword (e.g., 'train')
    for path in candidates:
        if keyword in os.path.basename(path):
            return path
    # Fallback: just return any folder with images
    return candidates[0] if candidates else None

# Auto‑detect the actual image directories
COCO_BASE = '/content/coco'
TRAIN_IMG_DIR = find_image_dir(COCO_BASE, 'train')
VAL_IMG_DIR   = find_image_dir(COCO_BASE, 'val')

print(f"📁 Detected TRAIN_IMG_DIR: {TRAIN_IMG_DIR}")
print(f"📁 Detected VAL_IMG_DIR:   {VAL_IMG_DIR}")

if TRAIN_IMG_DIR is None:
    raise FileNotFoundError("Could not find any folder with train images under /content/coco/")

# --- Load image metadata from the captions JSON ---
# train_ann was already defined earlier (captions file path)
with open(train_ann, 'r') as f:
    train_data = json.load(f)

# Try to find val captions file
ann_dir = '/content/coco/captions/annotations'
val_ann_path = glob.glob(os.path.join(ann_dir, '*captions_val20*.json'))
val_ann_path = val_ann_path[0] if val_ann_path else None
if val_ann_path:
    with open(val_ann_path, 'r') as f:
        val_data = json.load(f)
else:
    val_data = train_data

train_images = train_data['images']
val_images   = val_data['images'] if 'images' in val_data else train_images

# Build id -> filename mappings
train_id2file = {img['id']: img['file_name'] for img in train_images}
val_id2file   = {img['id']: img['file_name'] for img in val_images}

# --- Keep only images that exist on disk ---
def filter_existing(img_dir, ids, id2file):
    existing = []
    for img_id in ids:
        fname = id2file[img_id]
        path = os.path.join(img_dir, fname)
        if os.path.isfile(path):
            existing.append(img_id)
    return existing

all_train_ids = filter_existing(TRAIN_IMG_DIR, list(train_id2file.keys()), train_id2file)
all_val_ids   = filter_existing(VAL_IMG_DIR if VAL_IMG_DIR else TRAIN_IMG_DIR,
                                list(val_id2file.keys()), val_id2file)

print(f"✅ Train images found on disk: {len(all_train_ids)}")
if all_val_ids:
    print(f"✅ Val   images found on disk: {len(all_val_ids)}")
else:
    print("⚠️  No validation images found – using a subset of train images as validation.")
    all_val_ids = all_train_ids[:min(5000, len(all_train_ids))]

# --- 50% random subset (as allowed by the PDF) ---
random.seed(42)
random.shuffle(all_train_ids)
random.shuffle(all_val_ids)

train_ids_50 = all_train_ids[:len(all_train_ids)//2]
val_ids_50   = all_val_ids[:len(all_val_ids)//2]

print(f"📊 Final 50% subset — Train: {len(train_ids_50)} images, Val: {len(val_ids_50)} images")

✅ Device: cuda
📁 Detected TRAIN_IMG_DIR: /content/coco/train2014/train2014
📁 Detected VAL_IMG_DIR:   /content/coco/train2014/train2014
✅ Train images found on disk: 82783
⚠️  No validation images found – using a subset of train images as validation.
📊 Final 50% subset — Train: 41391 images, Val: 2500 images


In [ ]:
# ==========================================
# 6. Person 2 – Transforms & Dataset (uses detected paths)
# ==========================================
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class CocoImageDataset(Dataset):
    def __init__(self, img_dir, image_ids, id2file, transform):
        self.img_dir = img_dir          # must be the real folder with .jpg files
        self.image_ids = image_ids
        self.id2file = id2file
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        fname = self.id2file[img_id]
        path = os.path.join(self.img_dir, fname)   # this path must exist
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        return img, img_id

# Use the directories found in Cell 5
print(f"Using TRAIN_IMG_DIR: {TRAIN_IMG_DIR}")
print(f"Using VAL_IMG_DIR:   {VAL_IMG_DIR}")

train_dataset = CocoImageDataset(TRAIN_IMG_DIR, train_ids_50, train_id2file, transform)
val_dataset   = CocoImageDataset(VAL_IMG_DIR, val_ids_50, val_id2file, transform)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size:   {len(val_dataset)}")

Using TRAIN_IMG_DIR: /content/coco/train2014/train2014
Using VAL_IMG_DIR:   /content/coco/train2014/train2014
Train dataset size: 41391
Val dataset size:   2500


In [ ]:
# ==========================================
# 7. Person 2 – DataLoaders
# ==========================================
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 647 | Val batches: 40


In [ ]:
# ==========================================
# 8. Person 2 – ResNet50 (frozen, last FC removed)
# ==========================================
resnet = models.resnet50(pretrained=True)
# Remove the classification head → get 2048-dim features
modules = list(resnet.children())[:-1]
resnet_feat_extractor = torch.nn.Sequential(*modules).to(device)
resnet_feat_extractor.eval()  # inference mode
print("✅ ResNet50 feature extractor ready.")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 148MB/s]


✅ ResNet50 feature extractor ready.


In [ ]:
# ==========================================
# 🔧 FIX: Automatically locate the real train/val image folders
# ==========================================
import os

def find_images_folder(base):
    """
    Walk through 'base' and return the first folder that contains .jpg files.
    If base itself has .jpg files, return base.
    """
    if os.path.isdir(base) and any(f.endswith('.jpg') for f in os.listdir(base)):
        return base
    for root, dirs, files in os.walk(base):
        if any(f.endswith('.jpg') for f in files):
            return root
    return None

TRAIN_IMG_DIR = find_images_folder('/content/coco/train2014')
VAL_IMG_DIR   = find_images_folder('/content/coco/val2014')

# Fallback if val2014 does not exist or has no images
if VAL_IMG_DIR is None:
    print("⚠️  Validation image folder not found – using 10% of training images as validation.")
    # We'll just reuse train for validation (common practice)
    VAL_IMG_DIR = TRAIN_IMG_DIR
    # We will later select a separate set of IDs for validation from the training IDs
    # We already have all_val_ids from earlier, but we'll re‑assign them.
    # For now, just ensure VAL_IMG_DIR points to a folder with images.
    # We'll rebuild the val ids accordingly.
    use_train_as_val = True
else:
    use_train_as_val = False

print(f"✅ TRAIN_IMG_DIR: {TRAIN_IMG_DIR}")
print(f"✅ VAL_IMG_DIR:   {VAL_IMG_DIR}")

# Count files
train_files = [f for f in os.listdir(TRAIN_IMG_DIR) if f.endswith('.jpg')]
val_files   = [f for f in os.listdir(VAL_IMG_DIR) if f.endswith('.jpg')]
print(f"   Train images found: {len(train_files)}")
print(f"   Val images found:   {len(val_files)}")

# If we are using train as val, we must adjust val_ids_50 to be a distinct subset
# of the training IDs (not overlapping with train_ids_50).
# Let's re‑create the 50% subsets carefully from the existing filtered IDs.
# But we already have train_ids_50 and val_ids_50 from Cell 5 (which might be empty for val).
# Let's rebuild them here based on the actual files we can use.
# We'll re‑filter train_ids_50 to only those present in TRAIN_IMG_DIR, and create val from train if needed.

# Re‑create the full mapping for train (already done earlier, but we'll re‑fetch)
import json, glob, random
with open(train_ann, 'r') as f:
    train_data = json.load(f)
train_images = train_data['images']
train_id2file = {img['id']: img['file_name'] for img in train_images}

# Filter by existence in TRAIN_IMG_DIR
def filter_existing(img_dir, ids, id2file):
    return [i for i in ids if os.path.isfile(os.path.join(img_dir, id2file[i]))]

all_train_ids = filter_existing(TRAIN_IMG_DIR, list(train_id2file.keys()), train_id2file)
print(f"Total train IDs with valid images: {len(all_train_ids)}")

random.seed(42)
random.shuffle(all_train_ids)

# Split into train and validation (80/20 or 90/10)
split_idx = int(len(all_train_ids) * 0.8)
train_ids_50 = all_train_ids[:split_idx]
val_ids_50   = all_train_ids[split_idx:]

# If we actually have a separate val folder and we already had val IDs, we could use those.
# But to be safe and avoid missing files, we'll just use the train split for validation.
if not use_train_as_val:
    # Try to load val IDs from the val captions
    val_ann_path = glob.glob('/content/coco/captions/annotations/*captions_val20*.json')
    if val_ann_path:
        with open(val_ann_path[0], 'r') as f:
            val_data = json.load(f)
        val_images = val_data['images']
        val_id2file = {img['id']: img['file_name'] for img in val_images}
        all_val_ids = filter_existing(VAL_IMG_DIR, list(val_id2file.keys()), val_id2file)
        random.shuffle(all_val_ids)
        val_ids_50 = all_val_ids[:len(all_val_ids)//2]
        # If val_ids_50 is empty, fallback to train split
        if len(val_ids_50) == 0:
            print("Val IDs empty, falling back to train split.")
            val_ids_50 = all_train_ids[split_idx:]

# Ensure both sets are non‑empty
if len(train_ids_50) == 0 or len(val_ids_50) == 0:
    raise RuntimeError("Empty image list – check image files.")

print(f"✅ Final train IDs: {len(train_ids_50)}, val IDs: {len(val_ids_50)}")

# Rebuild datasets and loaders with corrected paths
from torch.utils.data import DataLoader

train_dataset = CocoImageDataset(TRAIN_IMG_DIR, train_ids_50, train_id2file, transform)
val_dataset   = CocoImageDataset(VAL_IMG_DIR, val_ids_50, val_id2file if not use_train_as_val else train_id2file, transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f"✅ Loaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches")

⚠️  Validation image folder not found – using 10% of training images as validation.
✅ TRAIN_IMG_DIR: /content/coco/train2014/train2014
✅ VAL_IMG_DIR:   /content/coco/train2014/train2014
   Train images found: 82783
   Val images found:   82783
Total train IDs with valid images: 82783
✅ Final train IDs: 66226, val IDs: 16557
✅ Loaders ready: 1035 train batches, 259 val batches


In [ ]:
# ==========================================
# 🔧 Final fix: Rebuild loaders with num_workers=0 (Colab safe)
# ==========================================
from torch.utils.data import DataLoader

# Make sure train_dataset, val_dataset, train_ids_50, val_ids_50 exist
# (they should if you already ran the previous fix cell that re-created them)
try:
    # If datasets don't exist, re-create them using the corrected directories & IDs
    TRAIN_IMG_DIR = '/content/coco/train2014/train2014'
    VAL_IMG_DIR   = '/content/coco/train2014/train2014'  # We'll use the same split as before
    # In the last fix we built train_ids_50, val_ids_50, train_id2file, transform.
    # Those variables should be in memory. If not, run the previous fix cell again.
    train_dataset = CocoImageDataset(TRAIN_IMG_DIR, train_ids_50, train_id2file, transform)
    val_dataset   = CocoImageDataset(VAL_IMG_DIR, val_ids_50, train_id2file, transform)
except NameError:
    raise RuntimeError(
        "Required variables not found. Please run the previous fix cell first."
    )

# Create loaders with ZERO workers (this eliminates the multiprocessing error)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"✅ Loaders rebuilt: {len(train_loader)} train batches, {len(val_loader)} val batches (num_workers=0)")

✅ Loaders rebuilt: 1035 train batches, 259 val batches (num_workers=0)


In [ ]:
# ==========================================
# 9. Person 2 – Extract Features
# ==========================================
def extract_features(loader, model, device):
    all_feat, all_ids = [], []
    with torch.no_grad():
        for imgs, ids in tqdm(loader, desc="Extracting"):
            imgs = imgs.to(device)
            feat = model(imgs)                     # (B, 2048, 1, 1)
            feat = feat.view(feat.size(0), -1)     # (B, 2048)
            all_feat.append(feat.cpu())
            all_ids.extend(ids)
    return torch.cat(all_feat, dim=0), all_ids

print("Extracting training features ...")
train_feat, train_ids_list = extract_features(train_loader, resnet_feat_extractor, device)
print("Extracting validation features ...")
val_feat, val_ids_list = extract_features(val_loader, resnet_feat_extractor, device)

print(f"Train features: {train_feat.shape} | Val features: {val_feat.shape}")

Extracting training features ...


Extracting: 100%|██████████| 1035/1035 [14:29<00:00,  1.19it/s]


Extracting validation features ...


Extracting: 100%|██████████| 259/259 [03:35<00:00,  1.20it/s]

Train features: torch.Size([66226, 2048]) | Val features: torch.Size([16557, 2048])


In [ ]:
# ==========================================
# 10. Person 2 – Save Features for Person 3
# ==========================================
import torch
import numpy as np

# Save as PyTorch .pt files (primary format for Person 3)
torch.save({'features': train_feat, 'image_ids': train_ids_list},
           '/content/train_features.pt')
torch.save({'features': val_feat,   'image_ids': val_ids_list},
           '/content/val_features.pt')

# Also save as .npy for easy inspection / compatibility
np.save('/content/train_features.npy', train_feat.numpy())
np.save('/content/val_features.npy', val_feat.numpy())

print("✅ Features saved:")
print("   /content/train_features.pt  (.npy)")
print("   /content/val_features.pt   (.npy)")

✅ Features saved:
   /content/train_features.pt  (.npy)
   /content/val_features.pt   (.npy)


In [ ]:
# ==========================================
# 11. Person 2 – (Optional) Auxiliary Classification
# ==========================================
import torch.nn as nn

class ClassifierHead(nn.Module):
    """
    A simple linear classifier on top of the 2048‑dim features.
    This is an auxiliary task to sharpen visual features (optional per the PDF).
    For COCO there are 80 object categories.
    """
    def __init__(self, feat_dim=2048, num_classes=80):
        super().__init__()
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

# Instantiate and move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clf_head = ClassifierHead().to(device)
print("✅ Auxiliary classifier defined (80 COCO classes).")

✅ Auxiliary classifier defined (80 COCO classes).


# Person 3 code

In [ ]:
# ==========================================
# 12. Person 3 – Reload Tokenizer & Align Captions (with type fix)
# ==========================================
import torch, pickle, json, string, os, glob
import numpy as np

# ---------- 1. Reload tokenizer ----------
with open('/content/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)
with open('/content/max_sequence_length.txt', 'r') as f:
    max_seq_len = int(f.read().strip())

vocab_size = len(tokenizer.word_index) + 1
start_idx = tokenizer.word_index['startseq']
end_idx   = tokenizer.word_index['endseq']
print(f"Vocab size: {vocab_size}, start: {start_idx}, end: {end_idx}")

# ---------- 2. Load features ----------
train_data = torch.load('/content/train_features.pt')
val_data   = torch.load('/content/val_features.pt')
train_feats = train_data['features']
train_ids   = train_data['image_ids']
val_feats   = val_data['features']
val_ids     = val_data['image_ids']
print(f"Train features: {train_feats.shape}, IDs: {len(train_ids)}")
print(f"Val features:   {val_feats.shape}, IDs: {len(val_ids)}")

# ---------- 3. Find caption file ----------
ANN_DIR = '/content/coco/captions/annotations'
train_ann = glob.glob(os.path.join(ANN_DIR, '*captions_train20*.json'))
if not train_ann:
    train_ann = glob.glob('/content/coco/**/*captions_train20*.json', recursive=True)
train_ann = train_ann[0] if train_ann else None
if not train_ann:
    raise FileNotFoundError("Caption file not found.")
print(f"Using caption file: {train_ann}")

# ---------- 4. Load annotations ----------
with open(train_ann, 'r') as f:
    ann_data = json.load(f)
annotations = ann_data['annotations']
print(f"Total captions: {len(annotations)}")

# ---------- 5. Clean function ----------
def clean_caption(caption):
    caption = caption.lower()
    caption = caption.translate(str.maketrans('', '', string.punctuation))
    caption = ''.join(ch for ch in caption if not ch.isdigit())
    caption = ' '.join(caption.split())
    return 'startseq ' + caption + ' endseq'

# ---------- 6. Diagnose type mismatch ----------
# Inspect a few IDs from both sources
print("\n--- Type Diagnosis ---")
print(f"Sample train_ids (first 5): {train_ids[:5]}")
print(f"Type of first train_id: {type(train_ids[0])}")
ann_ids_sample = [ann['image_id'] for ann in annotations[:5]]
print(f"Sample annotation image IDs (first 5): {ann_ids_sample}")
print(f"Type of first ann image_id: {type(ann_ids_sample[0])}")

# Force everything to int
train_ids_int = [int(i) for i in train_ids]
val_ids_int   = [int(i) for i in val_ids]

train_set_ids = set(train_ids_int)
val_set_ids   = set(val_ids_int)
ann_ids = set(ann['image_id'] for ann in annotations)

print(f"Intersection (train vs ann): {len(train_set_ids & ann_ids)}")
print(f"Intersection (val vs ann):   {len(val_set_ids & ann_ids)}")

# ---------- 7. Build aligned pairs using int IDs ----------
train_id2feat = {i: f for i, f in zip(train_ids_int, train_feats)}
val_id2feat   = {i: f for i, f in zip(val_ids_int, val_feats)}

train_pairs = []
val_pairs   = []
for ann in annotations:
    img_id = ann['image_id']          # int
    caption = clean_caption(ann['caption'].strip())
    seq = tokenizer.texts_to_sequences([caption])[0]
    if len(seq) < 2:
        continue
    if img_id in train_set_ids:
        train_pairs.append((train_id2feat[img_id], seq))
    elif img_id in val_set_ids:
        val_pairs.append((val_id2feat[img_id], seq))

print(f"\n✅ Train pairs built: {len(train_pairs)}")
print(f"✅ Val pairs built:   {len(val_pairs)}")

if len(train_pairs) == 0:
    raise RuntimeError("Still no training pairs. Check the JSON content and feature file consistency.")

Vocab size: 24408, start: 3, end: 4
Train features: torch.Size([66226, 2048]), IDs: 66226
Val features:   torch.Size([16557, 2048]), IDs: 16557
Using caption file: /content/coco/captions/annotations/captions_train2014.json
Total captions: 414113

--- Type Diagnosis ---
Sample train_ids (first 5): [tensor(236048), tensor(612), tensor(549556), tensor(182201), tensor(220654)]
Type of first train_id: <class 'torch.Tensor'>
Sample annotation image IDs (first 5): [318556, 116100, 318556, 116100, 379340]
Type of first ann image_id: <class 'int'>
Intersection (train vs ann): 66226
Intersection (val vs ann):   16557

✅ Train pairs built: 331287
✅ Val pairs built:   82826


In [ ]:
# ==========================================
# 13. Person 3 – CaptionDataset for Teacher Forcing
# ==========================================
from torch.utils.data import Dataset, DataLoader

class CaptionDataset(Dataset):
    def __init__(self, pairs, max_cap_len):
        self.samples = []
        for feat, full_seq in pairs:
            inp_seq  = full_seq[:-1]   # all tokens except the last (end token)
            targ_seq = full_seq[1:]    # all tokens except the first (start token)
            self.samples.append((feat, inp_seq, targ_seq))
        self.max_cap_len = max_cap_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        feat, inp, targ = self.samples[idx]
        # Pad to max_cap_len with 0 (padding index)
        inp  = inp  + [0] * (self.max_cap_len - len(inp))
        targ = targ + [0] * (self.max_cap_len - len(targ))
        return feat, torch.tensor(inp, dtype=torch.long), torch.tensor(targ, dtype=torch.long)

# Compute the maximum caption input/target length
max_cap_len = max(len(seq) - 1 for _, seq in train_pairs)
print(f"Maximum caption input/target length: {max_cap_len}")

# Create datasets
train_ds = CaptionDataset(train_pairs, max_cap_len)
val_ds   = CaptionDataset(val_pairs, max_cap_len)

# Create loaders (num_workers=0 for Colab safety)
BATCH_SIZE = 64
train_ldr = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_ldr   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_ldr)} | Val batches: {len(val_ldr)}")

Maximum caption input/target length: 50
Train batches: 5177 | Val batches: 1295


In [ ]:
# ==========================================
# 14. Person 3 – CNN‑RNN Model (Show & Tell)
# ==========================================
import torch.nn as nn

class ImageCaptioningModel(nn.Module):
    def __init__(self, feat_dim, embed_dim, hidden_dim, vocab_size, max_cap_len):
        super().__init__()
        # Project image feature to embedding space
        self.image_proj = nn.Linear(feat_dim, embed_dim)
        # Word embedding (padding_idx=0 so pad tokens are ignored)
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # LSTM decoder
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, batch_first=True)
        # Final dense layer to predict next word
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.max_cap_len = max_cap_len

    def forward(self, image_feats, captions):
        """
        image_feats: (batch, 2048)
        captions:    (batch, max_cap_len) – teacher-forced input (no start token)
        returns scores: (batch, max_cap_len, vocab_size)
        """
        # 1. Image embedding → (batch, 1, embed_dim)
        img_emb = self.image_proj(image_feats).unsqueeze(1)

        # 2. Word embeddings → (batch, max_cap_len, embed_dim)
        word_emb = self.embed(captions)

        # 3. Concatenate: image embedding first, then word tokens
        lstm_input = torch.cat([img_emb, word_emb], dim=1)   # (batch, 1+max_cap_len, embed_dim)

        # 4. LSTM forward pass
        lstm_out, _ = self.lstm(lstm_input)                  # (batch, 1+max_cap_len, hidden_dim)

        # 5. Discard the image timestep (first position)
        outputs = lstm_out[:, 1:, :]                         # (batch, max_cap_len, hidden_dim)

        # 6. Map to vocabulary logits
        scores = self.fc(outputs)                            # (batch, max_cap_len, vocab_size)
        return scores

# Hyperparameters
FEAT_DIM    = 2048      # ResNet50 output size
EMBED_DIM   = 256
HIDDEN_DIM  = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageCaptioningModel(FEAT_DIM, EMBED_DIM, HIDDEN_DIM, vocab_size, max_cap_len).to(device)
print(model)
print(f"✅ Model initialised on {device}")

ImageCaptioningModel(
  (image_proj): Linear(in_features=2048, out_features=256, bias=True)
  (embed): Embedding(24408, 256, padding_idx=0)
  (lstm): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=24408, bias=True)
)
✅ Model initialised on cuda


In [ ]:
# ==========================================
# 15. Person 3 – Training the Model
# ==========================================
import torch.optim as optim

criterion = nn.CrossEntropyLoss(ignore_index=0)   # ignore padding index 0
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for feats, inps, targs in tqdm(loader, desc="Training"):
        feats = feats.to(device)
        inps  = inps.to(device)
        targs = targs.to(device)

        optimizer.zero_grad()
        outputs = model(feats, inps)                        # (B, max_cap_len, V)
        loss = criterion(outputs.view(-1, vocab_size), targs.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for feats, inps, targs in tqdm(loader, desc="Validating"):
            feats = feats.to(device)
            inps  = inps.to(device)
            targs = targs.to(device)
            outputs = model(feats, inps)
            loss = criterion(outputs.view(-1, vocab_size), targs.view(-1))
            total_loss += loss.item()
    return total_loss / len(loader)

EPOCHS = 15
best_val_loss = float('inf')
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_ldr, optimizer, criterion, device)
    val_loss   = val_epoch(model, val_ldr, criterion, device)
    scheduler.step(val_loss)

    print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/content/best_image_captioning_model.pt')
        print("   -> Best model saved.")

# Save final model
torch.save(model.state_dict(), '/content/final_image_captioning_model.pt')
print("✅ Training complete. Models saved.")

Validating: 100%|██████████| 1295/1295 [00:45<00:00, 28.28it/s]


Epoch  1/15 | Train Loss: 3.0721 | Val Loss: 2.6782
   -> Best model saved.


Validating: 100%|██████████| 1295/1295 [00:48<00:00, 26.83it/s]


Epoch  2/15 | Train Loss: 2.5014 | Val Loss: 2.5800
   -> Best model saved.


Validating: 100%|██████████| 1295/1295 [00:46<00:00, 27.91it/s]


...

Epoch 13/15 | Train Loss: 1.6641 | Val Loss: 2.6884


Validating: 100%|██████████| 1295/1295 [00:46<00:00, 28.07it/s]


Epoch 14/15 | Train Loss: 1.6489 | Val Loss: 2.6979


Validating: 100%|██████████| 1295/1295 [00:45<00:00, 28.18it/s]


Epoch 15/15 | Train Loss: 1.6364 | Val Loss: 2.7076
✅ Training complete. Models saved.


In [ ]:
# ==========================================
# 16. Person 3 – Generate Captions Word‑by‑Word
# ==========================================
def generate_caption(model, image_feat, tokenizer, max_len, device):
    model.eval()
    with torch.no_grad():
        img_feat = image_feat.unsqueeze(0).to(device)          # (1, 2048)
        input_seq = torch.tensor([[start_idx]], dtype=torch.long).to(device)  # (1, 1)
        caption = []

        for _ in range(max_len):
            # Build input: image embedding + current word sequence
            img_emb = model.image_proj(img_feat).unsqueeze(1)  # (1, 1, E)
            word_emb = model.embed(input_seq)                 # (1, seq_len, E)
            lstm_in = torch.cat([img_emb, word_emb], dim=1)
            lstm_out, _ = model.lstm(lstm_in)
            # Take the output of the last timestep
            last_out = lstm_out[:, -1, :]                     # (1, H)
            scores = model.fc(last_out)
            _, predicted = scores.max(1)
            next_word = predicted.item()

            if next_word == end_idx:
                break

            caption.append(tokenizer.index_word.get(next_word, '<OOV>'))
            # Append predicted word for next step
            input_seq = torch.cat([input_seq, predicted.unsqueeze(0)], dim=1)

    return ' '.join(caption)

# Test generation on a random validation image
sample_feat, sample_id = val_feats[0], val_ids[0]
gen_cap = generate_caption(model, sample_feat, tokenizer, max_cap_len, device)
print(f"🖼️ Image ID {sample_id}: {gen_cap}")

🖼️ Image ID 158051: a woman is sitting at a table with a cake


In [ ]:
# ==========================================
# 17. Show More Generated Captions
# ==========================================
for i in range(5):
    feat = val_feats[i]
    img_id = val_ids[i]
    cap = generate_caption(model, feat, tokenizer, max_cap_len, device)
    print(f"Image {img_id}: {cap}")

Image 158051: a woman is sitting at a table with a cake
Image 379313: a laptop computer sitting on top of a wooden desk
Image 351268: a fire hydrant on the side of the street
Image 426022: a bathroom with a sink and a mirror
Image 148229: a man is standing on a surfboard in the water
